# Local Support Assembly Demo

This notebook demonstrates exact support-closure local assembly for element-partitioned finite element matrices. The METIS partition provides the core element sets; the local support patch used for assembly is then determined by DoF support.


In [ ]:
import sys
from pathlib import Path

import numpy as np
from ngsolve import *
from ngsolve.webgui import Draw
from netgen.geom2d import unit_square

import myassembling
print(myassembling)
print(myassembling.__file__)
print(dir(myassembling))

SetNumThreads(1)

## Mesh, Space, And METIS Element Partition

We build an `H1` space on the unit square and define the Poisson bilinear form with `SymbolicBFI(grad(u) * grad(v))`. The element partition is produced by METIS on the shared-DoF element graph. `core_partition[i]` is the non-overlapping METIS subdomain, while `partition[i]` is the optional graph-expanded element set returned by the helper. With `overlap_width=0`, these two sets coincide.


In [ ]:
mesh = Mesh(unit_square.GenerateMesh(maxh=0.12, quad_dominated=False))
fes = H1(mesh, order=1, dirichlet="left|right|bottom|top")
u, v = fes.TnT()
bfi = SymbolicBFI(grad(u) * grad(v))

# # Option 1 - Manual
# partition = [[], [], []]
# for el in mesh.Elements(VOL):
#     pts = [mesh[v].point for v in el.vertices]
#     cx = sum(p[0] for p in pts) / len(pts)
#     part = min(2, int(3 * cx))
#     partition[part].append(el.nr)


# Option 2 - Metis
from pymetis import part_graph
from ngsolve import VOL

def metis_partition_from_fes(fes, nparts, overlap_width=0, free_dofs_only=False):
    """
    METIS partition of elements using the shared-DoF element graph.

    Parameters
    ----------
    fes:
        NGSolve finite element space.

    nparts:
        Number of METIS subdomains.

    overlap_width:
        Number of overlap layers in the shared-DoF element graph.
        overlap_width=0 gives a non-overlapping METIS partition.

    free_dofs_only:
        If True, build the element graph only through free DoFs.
        This ignores Dirichlet-fixed DoFs when defining adjacency.

    Returns
    -------
    core_partition:
        Non-overlapping METIS partition.
        core_partition[p] is a list of element numbers.

    overlapping_partition:
        Overlapping partition after shared-DoF graph expansion.
        overlapping_partition[p] is a list of element numbers.

    cutcount:
        METIS cut count.
    """

    mesh = fes.mesh
    elements = list(mesh.Elements(VOL))
    ne = len(elements)

    if overlap_width < 0:
        raise ValueError("overlap_width must be >= 0")

    # Optional: restrict graph construction to free DoFs only
    freedofs = fes.FreeDofs() if free_dofs_only else None

    # Build dof -> elements table
    dof_to_elements = {}

    for i, el in enumerate(elements):
        for d in fes.GetDofNrs(el):
            d = int(d)
            if d < 0:
                continue
            if free_dofs_only and not freedofs[d]:
                continue

            dof_to_elements.setdefault(d, []).append(i)

    # Build element adjacency graph:
    # two elements are adjacent if they share at least one selected DoF
    adj = [set() for _ in range(ne)]

    for elems in dof_to_elements.values():
        for a in range(len(elems)):
            i = elems[a]
            for b in range(a + 1, len(elems)):
                j = elems[b]
                adj[i].add(j)
                adj[j].add(i)

    adjacency = [sorted(a) for a in adj]

    # METIS non-overlapping core partition
    cutcount, part_ids = part_graph(nparts, adjacency=adjacency)

    core_partition_pos = [[] for _ in range(nparts)]
    for i, part_id in enumerate(part_ids):
        core_partition_pos[part_id].append(i)

    # Expand each part by overlap_width layers in the same shared-DoF graph
    overlapping_partition_pos = []

    for part in core_partition_pos:
        patch = set(part)

        for _ in range(overlap_width):
            new_patch = set(patch)
            for i in patch:
                new_patch.update(adj[i])
            patch = new_patch

        overlapping_partition_pos.append(sorted(patch))

    # Convert internal element positions back to NGSolve element numbers
    core_partition = [
        [elements[i].nr for i in part]
        for part in core_partition_pos
    ]

    overlapping_partition = [
        [elements[i].nr for i in part]
        for part in overlapping_partition_pos
    ]

    return core_partition, overlapping_partition, cutcount
# print("ne =", mesh.ne, "ndof =", fes.ndof)
# print("core element counts =", [len(p) for p in partition])

core_partition, partition, cutcount = metis_partition_from_fes(
    fes,
    nparts=3,
    overlap_width=0,
    free_dofs_only=False,
)

print("ne =", mesh.ne, "ndof =", fes.ndof)
print("METIS cutcount =", cutcount)
print("core element counts =", [len(p) for p in core_partition])
print("overlapping element counts =", [len(p) for p in partition])



## Visualize The METIS Subdomains

This block draws one scalar element marker field per subdomain. Elements in `core_partition[i]` are marked with value `1.0`. Elements that would be present only in the graph-expanded `partition[i]` are marked with value `1.5`. For the current `overlap_width=0` setting, the plot shows only the non-overlapping METIS core elements.


In [ ]:
l2 = L2(mesh, order=0)

for i, elements in enumerate(partition):
    omega_i = GridFunction(l2, name=f"Omega_{i}")
    omega_i.vec[:] = 0

    core_set = set(core_partition[i])
    overlap_set = set(elements)

    # mark overlapping-only elements first
    for elnr in overlap_set - core_set:
        ei = ElementId(VOL, elnr)
        omega_i.vec[l2.GetDofNrs(ei)[0]] = 1.5

    # mark core elements on top
    for elnr in core_set:
        ei = ElementId(VOL, elnr)
        omega_i.vec[l2.GetDofNrs(ei)[0]] = 1.0

    Draw(omega_i, mesh, f"subdomain {i}: core=1, overlap=1.5")

## Bilinear Form Integrator

`bfi` is a generic NGSolve `BilinearFormIntegrator`. The local assembler does not know the PDE by itself; it only asks the integrator to compute each element matrix. `SymbolicBFI(grad(u) * grad(v))` defines the Poisson stiffness integrator using NGSolve built-ins. This keeps `myassembling.cpp` focused on support-patch construction and local assembly, without a custom Laplace integrator.


## Assemble Local Support Matrices

For each input element set in `partition`, the C++ code builds the exact support closure needed to reproduce the global core-core matrix block. In the current demo `partition` is the METIS core partition because `overlap_width=0`.

- `core_elements`: the input elements for this local problem
- `core_dofs`: all global DoFs appearing on `core_elements`
- `support_elements`: all elements whose DoF list intersects `core_dofs`
- `support_dofs`: all global DoFs appearing on `support_elements`
- `core_in_support`: local indices of `core_dofs` inside `support_dofs`

The local matrix is assembled only over `support_elements` using `support_dofs` as local numbering.


In [ ]:
patches = myassembling.MyAssembleLocalSupportMatrices(fes, bfi, partition)

for i, patch in enumerate(patches):
    print(f"Omega_{i}:")
    print("  core_elements   =", len(patch.core_elements))
    print("  support_elements=", len(patch.support_elements))
    print("  core_dofs       =", len(patch.core_dofs))
    print("  support_dofs    =", len(patch.support_dofs))
    print("  core_in_support =", list(patch.core_in_support))
    print("  matrix shape    =", (patch.mat.height, patch.mat.width))

    for k, g in enumerate(patch.core_dofs):
        assert patch.support_dofs[patch.core_in_support[k]] == g

## Visualize The Support Closure

This block draws one support-closure patch per local matrix. Value `1` marks the input core elements for the patch. Value `2` marks additional support elements: elements outside the core that still contribute to the core-core block because they share at least one DoF with `core_dofs`. Value `0` marks all unrelated elements.


In [ ]:
for patch_id in range(len(patches)):
    patch = patches[patch_id]

    patch_view = GridFunction(l2, name="support_patch_view")
    patch_view.vec[:] = 0
    for elnr in patch.support_elements:
        patch_view.vec[l2.GetDofNrs(ElementId(VOL, elnr))[0]] = 2
    for elnr in patch.core_elements:
        patch_view.vec[l2.GetDofNrs(ElementId(VOL, elnr))[0]] = 1

    Draw(patch_view, mesh, f"Omega_{patch_id} core and support closure")

## Verify Against Global Assembly

The intended correctness condition is only the core-core block:

`A_support[core_in_support, core_in_support] == A_global[core_dofs, core_dofs]`

This is why the support-closure plot includes elements marked `2`: those elements are necessary for exact local reassembly of entries involving the core DoFs. We do not compare the full `A_support` matrix with `A_global[support_dofs, support_dofs]`; that equality is not required.


In [ ]:
A_global = myassembling.MyAssembleMatrix(fes, bfi)

def dense_submatrix(mat, rows, cols):
    return np.array([[mat[i, j] for j in cols] for i in rows], dtype=float)

for i, patch in enumerate(patches):
    A_support = np.array(patch.mat.ToDense(), dtype=float)
    ids = list(patch.core_in_support)
    A_core_from_support = A_support[np.ix_(ids, ids)]
    A_core_from_global = dense_submatrix(A_global, patch.core_dofs, patch.core_dofs)
    err = np.linalg.norm(A_core_from_support - A_core_from_global, ord=np.inf)
    print(f"Omega_{i}: ||A_support[core,core] - A_global[core,core]||_inf = {err:.3e}")
    assert err < 1e-12